In [1]:
!pip install -q accelerate peft bitsandbytes transformers trl

## Fine-Tuning LLaMA 2

In [14]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from transformers import BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer # Supervised fine-tuning
from huggingface_hub import notebook_login , login

In [3]:
# Set environment variables for memory efficiency
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Enable memory-efficient attention mechanisms
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_flash_sdp(True)

In [5]:
# Model configuration
base_model = "NousResearch/Llama-2-7b-chat-hf"
df_name = "mlabonne/guanaco-llama2-1k"
new_model = "llama-2-7b-chat-guanaco"
compute_dtype = getattr(torch, "float16")


# Quantization configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,  # Enable double quantization for memory efficiency
)


In [6]:
# Load the model with quantization settings
model = AutoModelForCausalLM.from_pretrained(
    base_model ,
    quantization_config = quant_config ,
    device_map={"": 0}
)

model.config.use_cache = False
model.config.pretraining_tp = 1

# Load and configure tokenizer

tokenizer = AutoTokenizer.from_pretrained(base_model , trust_remote_code = True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [7]:
df = load_dataset(df_name)

if "train" in df :
  df_train = df["train"]

else :
  df_train = df


README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

(…)-00000-of-00001-9ad84bb9cf65a42f.parquet:   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
# LoRA configuration with reduced parameters for memory efficiency
peft_conf = LoraConfig(
    lora_alpha = 16 ,
    lora_dropout = 0.1 ,
    r = 16 ,
    bias = "none" ,
    task_type = "CAUSAL_LM"
)

In [9]:
# Training arguments with reduced batch size and increased gradient accumulation
training_params = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=1,  # Reduced batch size for memory efficiency
    gradient_accumulation_steps=4,  # Increased to compensate for smaller batch size
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,  # Enable mixed precision for memory efficiency
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    report_to=['none']
)

trainer = SFTTrainer(
    model=model,
    args=training_params,
    train_dataset=df_train,
    peft_config=peft_conf,
)

# Start training
print("Starting training...")
trainer.train()

Converting train dataset to ChatML:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Starting training...


Step,Training Loss
25,1.345900
50,1.619300
75,1.211300
100,1.432900
125,1.174900
150,1.355600
175,1.168800
200,1.448300


Step,Training Loss
25,1.345900
50,1.619300
75,1.211300
100,1.432900
125,1.174900
150,1.355600
175,1.168800
200,1.448300
225,1.153300
250,1.516000


TrainOutput(global_step=250, training_loss=1.3426305923461914, metrics={'train_runtime': 1403.4795, 'train_samples_per_second': 0.713, 'train_steps_per_second': 0.178, 'total_flos': 1.673178221039616e+16, 'train_loss': 1.3426305923461914})

In [10]:
# Save the model
print(f"Saving model to {new_model}...")
trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)
print("Training complete and model saved!")

Saving model to llama-2-7b-chat-guanaco...
Training complete and model saved!


In [17]:
# login to HF
notebook_login()


In [35]:
from huggingface_hub import login, whoami

login()

#Confirm identity
user_info = whoami()
hf_username = user_info['name']

#Create repo name based on your username
base_model = "meta-llama/Llama-2-7b-chat-hf"
new_model = "llama-2-7b-chat-guanaco"
output_dir = f"{base_model.split('/')[-1]}-{new_model}"
repo_name = f"{hf_username}/{output_dir}"

print(f"Pushing model to: {repo_name}")

#Push model and tokenizer
trainer.model.push_to_hub(repo_name, commit_message="Training Complete")
tokenizer.push_to_hub(repo_name, commit_message="Training Complete")

Pushing model to: SayedAli1/Llama-2-7b-chat-hf-llama-2-7b-chat-guanaco


adapter_model.safetensors:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/SayedAli1/Llama-2-7b-chat-hf-llama-2-7b-chat-guanaco/commit/6bf283c317f4172d644a411de08b060fc4a10691', commit_message='Training Complete', commit_description='', oid='6bf283c317f4172d644a411de08b060fc4a10691', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SayedAli1/Llama-2-7b-chat-hf-llama-2-7b-chat-guanaco', endpoint='https://huggingface.co', repo_type='model', repo_id='SayedAli1/Llama-2-7b-chat-hf-llama-2-7b-chat-guanaco'), pr_revision=None, pr_num=None)

## Inference

In [37]:
from peft import PeftModel
from pprint import pprint
from transformers import Pipeline, pipeline


In [38]:
print("Loading model for inference...")
model_path = "SayedAli1/Llama-2-7b-chat-hf-llama-2-7b-chat-guanaco"
tokenizer_path = "SayedAli1/Llama-2-7b-chat-hf-llama-2-7b-chat-guanaco"

base_model = "NousResearch/Llama-2-7b-chat-hf"


# Load base model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quant_config,
    device_map={"": 0}
)


# Apply the fine-tuned LoRA weights
model = PeftModel.from_pretrained(model, model_path)
model.eval()  # Set to evaluation mode

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

# Create a text generation pipeline
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto"
)



Loading model for inference...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'Gemma3ForConditionalGeneration', 'Gemma3ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'Glm4ForCausalLM', 'GotOcr2ForConditionalGeneration', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoFo

In [39]:
# Define a helper function for inference
def generate_response(prompt, max_new_tokens=100):
    result = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    return result[0]['generated_text']

def format_prompt(instruction, input_text=""):
    return f"<s>[INST] {instruction} {input_text} [/INST]"

prompt = format_prompt("What is the capital of France?")
response = generate_response(prompt)
print(response)

<s>[INST] What is the capital of France?  [/INST] The capital of France is Paris.


In [44]:
prompt = format_prompt("Who is the President of Egypt? ")
response = generate_response(prompt)
print(response)

<s>[INST] Who is the President of Egypt?   [/INST] The current president of Egypt is Abdel Fattah el-Sisi.


# End